# VoxTell on AMOS22 MRI (Kaggle GPU)

Runs the same positional-bias evaluation already done with Text3DSAM and Medical-SAM3,
but with [VoxTell](https://github.com/MIC-DKFZ/VoxTell) (CVPR 2026) — a free-text-promptable
universal 3D medical segmentation model, on your existing 10-subject AMOS22 MRI sample
already committed to the repo (no extra download, no full-dataset run).

**Scope of this notebook: AMOS22 MRI only.** Neurite OASIS is intentionally not run here —
duplicate this notebook and swap the `AMOS_DIR`/manifest cell to add it later.

**Key difference from the Medical-SAM3 notebook:** VoxTell accepts a **list of text prompts
in one call per volume** (all 13 AMOS organs at once), not one subprocess call per organ —
so this notebook makes 10 model calls total (one per subject), each producing 13 organ masks,
for 130 result rows total, matching your manifest exactly.

**Output:** one row per (subject, organ) in `predictions_voxtell_amos22_mri.csv`, plus one
3-panel PNG visualization (original / ground truth / prediction) per (subject, organ) under
`results_voxtell_amos/<subject_id>/<organ>.png`, plus the raw predicted mask array as an
`.npz` per (subject, organ) under `masks_voxtell_amos/<organ>/<subject_id>.npz` — see the
markdown cell right before the full-run cell for the exact CSV column list.

**Before running anything on Kaggle:**
- Notebook Settings -> Accelerator -> GPU (T4 x2 or P100)
- Add a Kaggle **Secret** named `GITHUB_PAT` (Add-ons -> Secrets) with a GitHub PAT that has
  read access to your private research repo. Do **not** hardcode a token in any cell — a
  previous notebook in this project had a live token committed directly in a cell, which is a
  real credential leak; if that token hasn't been revoked yet, revoke/rotate it now regardless
  of whether the repo is private.


In [ ]:
import subprocess, sys, re
from pathlib import Path

# This cell installs everything (voxtell + deps, a pinned transformers, and a clean
# torch+torchvision pair) BEFORE torch is ever imported in this kernel, then requires a
# kernel restart. Two things force this ordering:
#
# 1. `pip install voxtell` pins torch<2.9, so it can silently downgrade Kaggle's preinstalled
#    torch (observed both outcomes across runs -- sometimes torch stays put, sometimes it's
#    pulled to 2.8.x) as a side effect of installing voxtell itself.
# 2. Separately, Kaggle's preinstalled torch build has been observed shipping in a
#    self-inconsistent state regardless: Python files referencing C-level symbols (e.g.
#    torch.distributed.tensor._ops._embedding_ops._MaskPartial) that the paired compiled
#    extension doesn't actually provide -- surfaces only when importing torch's heavier
#    submodules (dynamo/FSDP), which VoxTell's predictor does via
#    `from torch._dynamo import OptimizedModule`.
#
# Both problems need a clean, paired reinstall of torch+torchvision (one pip command, no
# --no-deps, so pip picks a mutually ABI-compatible pair -- an earlier version of this fix
# reinstalled torch alone, which left the already-installed torchvision mismatched and broke
# with "RuntimeError: operator torchvision::nms does not exist"). But torch's C extension
# (`torch._C`) is a process-wide singleton: once loaded, Python can't safely swap its files
# out from under itself (tried purging sys.modules and re-importing in the same kernel --
# raises "RuntimeError: function '_has_torch_function' already has a docstring"). So this
# entire install+fix sequence has to run, and finish, WITHOUT importing torch anywhere in
# this kernel -- read the installed version via `pip show` (subprocess), never `import
# torch` -- and the kernel must be restarted afterward before anything imports torch.
_marker = Path('/kaggle/working/.env_setup_done')
if _marker.exists():
    print('environment already installed/fixed earlier in this session -- skipping. '
          f'({_marker.read_text()})')
else:
    def _pip(*args):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

    print('installing voxtell + deps...')
    _pip('voxtell', 'nibabel', 'pandas', 'matplotlib', 'huggingface_hub')

    _show = subprocess.run([sys.executable, '-m', 'pip', 'show', 'torch'],
                            capture_output=True, text=True, check=True).stdout
    _torch_version = re.search(r'^Version: (.+)$', _show, re.MULTILINE).group(1).split('+')[0]
    print(f'reinstalling torch=={_torch_version} + torchvision cleanly (this can take a minute)...')
    _pip('--force-reinstall', '--no-cache-dir', f'torch=={_torch_version}', 'torchvision')

    # transformers ALONE (no other top-level packages in the same command, no --no-deps) so
    # pip resolves its own tokenizers/huggingface_hub requirements together with it -- e.g.
    # transformers 4.57.6 needs huggingface-hub<1.0,>=0.34.0 and tokenizers<=0.23.0,>=0.22.0.
    # Bundling unconstrained tokenizers/huggingface_hub upgrades into a separate command (an
    # earlier version of this fix did that) pulled newer versions that then violated
    # transformers's own pins.
    print('reinstalling transformers pinned <5...')
    _pip('--force-reinstall', '--no-cache-dir', 'transformers>=4.54.0,<5')

    _marker.write_text(f'torch=={_torch_version}')
    raise RuntimeError(
        'Environment installed/fixed. RESTART THE KERNEL NOW (Kaggle: Run -> Restart '
        'session, or the restart icon in the toolbar), then re-run this notebook from the '
        'top -- this cell will then just print "skipping" and the rest of the notebook will '
        'run against the clean, consistent install. Do not run any later cell before '
        'restarting: torch would still be broken/mismatched in THIS kernel process even '
        'though the files on disk are now fixed.'
    )


## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU visible -- enable GPU under Notebook Settings -> Accelerator."
print("CUDA OK:", torch.cuda.get_device_name(0))


## 2. Setup Working Directories (Kaggle Environment)

In [ ]:
import os
from pathlib import Path

# /kaggle/working is the writable output directory that persists in Kaggle outputs
WORKING_DIR = Path('/kaggle/working')
RESULTS_ROOT = WORKING_DIR / 'results'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Results will be saved under:', RESULTS_ROOT)


## 3. Install VoxTell

VoxTell is pip-installable directly (no cloning/build-from-source, no pinned-numpy fights
like Medical-SAM3 needed). PyTorch is already present in the Kaggle GPU image, so we install
VoxTell without re-pulling torch.

In [ ]:
%cd /kaggle/working

# All installation/reinstallation (voxtell + deps, transformers pin, the torch+torchvision
# consistency fix) happens in the first cell of this notebook, BEFORE torch is ever imported
# in this kernel -- see that cell for why. This cell just verifies the result.
import torch
import transformers
import voxtell
print('transformers version:', transformers.__version__)
assert int(transformers.__version__.split('.')[0]) < 5,     f"transformers {transformers.__version__} violates voxtell's own transformers<5 pin -- re-run the first cell and restart the kernel."
print('voxtell installed:', voxtell.__file__)
print('torch version:', torch.__version__, '-- CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA not available -- check the first cell's pip output above."


## 4. Clone your research repo (AMOS22 MRI volumes/manifest)

Uses a Kaggle Secret instead of a hardcoded token — this cell never has a real token written
into the notebook file itself.

In [ ]:
import shutil
import subprocess

from kaggle_secrets import UserSecretsClient

_gh_token = UserSecretsClient().get_secret("GITHUB_PAT")
assert _gh_token, "Add a Kaggle Secret named GITHUB_PAT (Add-ons -> Secrets) before running this cell."

_clone_url = f'https://{_gh_token}@github.com/fatemafaria142/positional-bias-medical-vlms.git'
_dest = '/kaggle/working/research_repo'
shutil.rmtree(_dest, ignore_errors=True)

_result = subprocess.run(
    ['git', 'clone', '--branch', 'faria/3d-segmentation-datasets', '--depth', '1',
     _clone_url, _dest],
    capture_output=True, text=True,
)
_stderr_scrubbed = _result.stderr.replace(_gh_token, '***')
_token_len = len(_gh_token)
del _gh_token, _clone_url

if _result.returncode != 0:
    print('git clone FAILED (token length was', _token_len, 'chars):')
    print(_stderr_scrubbed)
else:
    print('cloned OK')

RESEARCH_REPO = Path('/kaggle/working/research_repo')
print('samples:', os.listdir(RESEARCH_REPO / 'samples'))


## 5. Download and load the VoxTell model

**Correction:** the published PyPI package (`voxtell`, v0.1.0 -- confirmed by checking the
actual tagged source on GitHub, since `pip install voxtell` does not track the `main` branch)
requires `model_dir` as a **required** argument to `VoxTellPredictor` -- there is no
auto-download-on-first-use in this released version (that convenience helper exists only on
GitHub's `main` branch, not yet in a PyPI release). So we download the checkpoint ourselves
first, the same way Medical-SAM3's notebook does its checkpoint fetch, then pass the resulting
local folder in as `model_dir`.

Checkpoint: [huggingface.co/mrokuss/VoxTell](https://huggingface.co/mrokuss/VoxTell), folder
`voxtell_v1.1` (contains `plans.json` and `fold_0/checkpoint_final.pth`).

In [ ]:
import torch
from huggingface_hub import snapshot_download
from voxtell.inference.predictor import VoxTellPredictor

VOXTELL_MODEL_VERSION = 'voxtell_v1.1'
CKPT_CACHE_DIR = WORKING_DIR / 'checkpoints'
CKPT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
_local_repo_dir = snapshot_download(
    repo_id='mrokuss/VoxTell',
    allow_patterns=[f'{VOXTELL_MODEL_VERSION}/*'],
    local_dir=str(CKPT_CACHE_DIR),
)
VOXTELL_MODEL_DIR = str(Path(_local_repo_dir) / VOXTELL_MODEL_VERSION)
print('VoxTell checkpoint at', VOXTELL_MODEL_DIR)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
predictor = VoxTellPredictor(model_dir=VOXTELL_MODEL_DIR, device=device)
print('VoxTell predictor ready on', device)


### 5b. Fix: text encoder OOM on a single T4/P100

VoxTell's text encoder is `Qwen/Qwen3-Embedding-4B` (4 billion parameters). The installed
v0.1.0 release loads it with `AutoModel.from_pretrained(text_encoding_model)`, no `torch_dtype`
given -- Hugging Face's default in that case is **full fp32**, so the text encoder alone is
~16GB, larger than a T4's entire 14.56GB usable memory, before the segmentation network even
factors in. This is why the smoke test OOM'd on `self.text_backbone.to(self.device)` --
it has nothing to do with passing 13 organs at once (the model dominates the memory cost, not
the number of prompts).

There's no public constructor argument to fix this, so this cell swaps in a half-precision
copy of the text encoder after the predictor is built (roughly halving its footprint to ~8GB,
which should coexist with the segmentation network), and wraps `embed_text_prompts` so its
output is always cast back to `float32` before being used further -- avoiding any fp16/fp32
dtype-mismatch risk in the segmentation network's fusion layers downstream. It reuses VoxTell's
own original `embed_text_prompts` method internally (just swaps the model and casts the
result), so none of its internal tokenization/pooling logic is duplicated here.

In [ ]:
import torch
from transformers import AutoModel

# Guarded so re-running this cell never re-wraps an already-wrapped method --
# the true original is captured on the predictor object itself, exactly once.
if not hasattr(predictor, '_orig_embed_text_prompts'):
    predictor._orig_embed_text_prompts = predictor.embed_text_prompts

predictor.text_backbone = AutoModel.from_pretrained(
    'Qwen/Qwen3-Embedding-4B', dtype=torch.float16  # VoxTell v0.1.0's default text encoder
).eval()

@torch.inference_mode()
def _embed_text_prompts_fp16_safe(text_prompts):
    embeddings = predictor._orig_embed_text_prompts(text_prompts)
    return embeddings.float()

predictor.embed_text_prompts = _embed_text_prompts_fp16_safe
print('text encoder swapped to fp16 (output still cast back to float32)')


## 6. Helpers (Dice / centroid / bbox / side, same logic as scripts/common3d.py)

In [ ]:
import csv, time
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from nnunetv2.imageio.nibabel_reader_writer import NibabelIOWithReorient

RESEARCH_REPO = Path(RESEARCH_REPO)
reader = NibabelIOWithReorient()

_OPPOSITE = {"R": "L", "L": "R", "A": "P", "P": "A", "S": "I", "I": "S"}

def load_volume_ras(path):
    """Load a NIfTI image via VoxTell's own reader (nnunetv2's
    NibabelIOWithReorient) so predicted masks and ground truth end up in the
    exact same RAS-reoriented array space. Returns the array UNSQUEEZED, in
    the (1, X, Y, Z) channel-first shape voxtell.inference.predictor expects
    -- confirmed against the real predict_single_image() source, which is
    called with this exact shape in VoxTell's own README example, not a
    squeezed 3D array."""
    data, props = reader.read_images([str(path)])
    return np.asarray(data), props


def spatial_view(arr_4d):
    """Drop the leading channel dim for slicing/plotting only -- never for
    what's actually passed to predictor.predict_single_image()."""
    a = np.asarray(arr_4d)
    while a.ndim > 3 and a.shape[0] == 1:
        a = a[0]
    return a

def load_seg_ras(path):
    """Ground truth is only ever compared voxel-wise, never fed to the model,
    so it's fine (and simpler) to squeeze this one down to (X, Y, Z) here."""
    data, props = reader.read_seg(str(path))
    return spatial_view(data).astype(np.int32), props

def mask_ground_truth_3d(mask: np.ndarray):
    """Centroid/bbox/side in RAS-reoriented voxel space -- axis 0/1/2 are
    always R/A/S respectively once reoriented, unlike the per-subject
    varying axis order recorded in the manifest (which was computed in each
    subject's ORIGINAL array order, not RAS)."""
    coords = np.nonzero(mask)
    if len(coords[0]) == 0:
        return None
    centroid = tuple(float(c.mean()) for c in coords)
    bbox = tuple(int(c.min()) for c in coords) + tuple(int(c.max()) for c in coords)
    axcodes = ("R", "A", "S")
    sides = [code if centroid_val > mask.shape[axis] / 2 else _OPPOSITE[code]
             for axis, (code, centroid_val) in enumerate(zip(axcodes, centroid))]
    return centroid, bbox, tuple(sides)

def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return 1.0 if denom == 0 else float(2.0 * inter / denom)

def struct_slug(name: str) -> str:
    return name.strip().lower().replace(' ', '_').replace('/', '-')

def window(img_slice):
    lo, hi = np.percentile(img_slice, [1, 99])
    if hi <= lo:
        return img_slice
    return np.clip((img_slice - lo) / (hi - lo), 0, 1)

def load_manifest_by_subject(manifest_path):
    subjects = {}
    with open(manifest_path, newline='') as f:
        for row in csv.DictReader(f):
            subjects.setdefault(row['subject_id'], []).append(row)
    return subjects

GT_COLOR = (0.13, 0.70, 0.30)
PRED_COLOR = (0.85, 0.15, 0.15)
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'axes.titlesize': 12, 'axes.titleweight': 'medium',
    'figure.titlesize': 13, 'legend.fontsize': 10,
    'savefig.facecolor': 'white',
})

def save_panel_figure(out_path, prompt_text, dice, img_slice, gt_slice, pred_slice):
    h, w = img_slice.shape
    fig, axes = plt.subplots(1, 3, figsize=(9.6, 9.6 * h / w / 3 + 0.9),
                              gridspec_kw={'wspace': 0.02})
    for ax, label in zip(axes, ('Original', 'Ground Truth', 'Prediction')):
        ax.imshow(img_slice, cmap='gray', origin='lower')
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_xlabel(label, fontsize=11, labelpad=6)
    gt_overlay = np.zeros((*gt_slice.shape, 4)); gt_overlay[gt_slice] = (*GT_COLOR, 0.5)
    axes[1].imshow(gt_overlay, origin='lower')
    pred_overlay = np.zeros((*pred_slice.shape, 4)); pred_overlay[pred_slice] = (*PRED_COLOR, 0.5)
    axes[2].imshow(pred_overlay, origin='lower')
    legend_handles = [
        plt.Line2D([0], [0], marker='s', color='none', markerfacecolor=GT_COLOR, markersize=12, label='Ground Truth'),
        plt.Line2D([0], [0], marker='s', color='none', markerfacecolor=PRED_COLOR, markersize=12, label='Prediction'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02), frameon=False)
    fig.suptitle(f'"{prompt_text}"', fontsize=13, fontweight='bold', y=1.06)
    ax_title_color = PRED_COLOR if dice < 0.3 else '#1a1a1a'
    axes[1].set_title(f'Dice = {dice:.3f}', fontsize=12, fontweight='bold', color=ax_title_color, pad=8)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.90, bottom=0.16, wspace=0.02)
    fig.savefig(out_path, dpi=200, bbox_inches='tight', pad_inches=0.15)
    plt.close(fig)

print('helpers ready')


## 7. Dataset config — AMOS22 MRI

Uses only your existing 10-subject sample from the GitHub repo (`samples/amos22_mri/`) —
this notebook does not download anything extra or reach for the full 60-subject dataset.

In [ ]:
AMOS_DIR = RESEARCH_REPO / 'samples' / 'amos22_mri'
MANIFEST_PATH = AMOS_DIR / 'manifest_amos22_mri.csv'

# None = run every subject present in the manifest. Set an int to cap it (e.g. for a quick test).
N_SUBJECTS = None

all_subjects = load_manifest_by_subject(MANIFEST_PATH)
if N_SUBJECTS is not None:
    all_subjects = dict(list(all_subjects.items())[:N_SUBJECTS])

n_organs_per_subject = len(next(iter(all_subjects.values())))
print(f'AMOS22 MRI: {len(all_subjects)} subject(s) in this run, '
      f'{n_organs_per_subject} organs/subject -> {len(all_subjects) * n_organs_per_subject} total (subject, organ) rows')
print('subjects:', list(all_subjects.keys()))


## 8. Smoke test — ONE subject, all its organs in ONE call

VoxTell takes a *list* of prompts per volume, so unlike Medical-SAM3 (one subprocess call
per organ), a single subject's all-13-organs prediction happens in one `predict_single_image`
call. Time this before committing to the full run — the next cell uses this number to
extrapolate the full-dataset runtime.

In [ ]:
smoke_subject_id, smoke_rows = next(iter(all_subjects.items()))
smoke_organs = [r['organ'] for r in smoke_rows]
smoke_image_path = AMOS_DIR / smoke_rows[0]['image_path']
smoke_label_path = AMOS_DIR / smoke_rows[0]['label_path']
print('smoke test subject:', smoke_subject_id, '-- organs:', smoke_organs)

t0 = time.time()
smoke_img, smoke_props = load_volume_ras(smoke_image_path)  # raw (1, X, Y, Z), as VoxTell expects
smoke_pred = predictor.predict_single_image(smoke_img, smoke_organs)  # -> (num_prompts, X, Y, Z), already binary 0/1
smoke_elapsed = time.time() - t0

print(f'smoke test elapsed: {smoke_elapsed:.1f}s for {len(smoke_organs)} organs '
      f'({smoke_elapsed / len(smoke_organs):.2f}s/organ if it scaled linearly, '
      'though in practice most of this cost is the ONE shared image encoding pass)')
print('input image shape (fed to model):', smoke_img.shape)
print('prediction array shape:', smoke_pred.shape, 'dtype:', smoke_pred.dtype,
      '-- already binary (0/1), no thresholding needed on our side')


## 9. Runtime estimate for your manifest

Measured from the smoke test above (not guessed): VoxTell encodes each volume once and
predicts all of that subject's organs from that one encoding, so runtime scales with
**subject count**, not (subjects x organs) the way the Medical-SAM3 per-organ loop did.

In [ ]:
ORGANS_PER_SUBJECT = n_organs_per_subject  # 13, verified in this MRI subset
per_subject_seconds = smoke_elapsed
current_manifest_subjects = len(all_subjects)

print(f'Measured: {per_subject_seconds:.1f}s for 1 subject ({ORGANS_PER_SUBJECT} organs) on this GPU.')
print()
print(f'Your manifest ({current_manifest_subjects} subjects, {current_manifest_subjects * ORGANS_PER_SUBJECT} rows): '
      f'~{per_subject_seconds * current_manifest_subjects / 60:.1f} min total, linearly extrapolated from this one subject.')
print()
print('Caveat: AMOS22 MRI volume shapes vary a lot across subjects (e.g. 260x80x320 up to')
print('512x468x72), so per-subject time will vary -- run the full loop below and check the')
print('actual per-subject timing it prints rather than trusting this single-subject estimate.')


## 10. Full run — one VoxTell call per subject (all organs at once)

**How results are saved:**

- **CSV** — `results/predictions_voxtell_amos22_mri.csv`, one row per (subject, organ) —
  130 rows total for your 10-subject sample. Columns:
  - `subject_id`, `split`, `organ`, `label_id`, `prompt_text` — which case this row is
  - `volume_shape` — that subject's volume shape, copied from the manifest
  - `n_pred_voxels`, `dice`, `inference_seconds` — how big the predicted mask was, its Dice
    against ground truth, and time spent (averaged across that subject's organs)
  - `pred_centroid_vox_0/1/2`, `pred_bbox_vox_{0,1,2}{min,max}`, `pred_side_axis0/1/2` —
    predicted mask's centroid, bounding box, and R/A/S side, all in RAS-reoriented voxel space
  - `gt_centroid_vox_0/1/2`, `gt_side_axis0/1/2` — ground truth, copied straight from your
    manifest (note: these were computed in each subject's *original* array order, not
    RAS-reoriented, so compare `pred_side_*` to `gt_side_*` as R/A/S labels, not raw indices)
  - Row is flushed to disk immediately after each organ finishes — safe to stop anytime
- **Visualizations** — `results/results_voxtell_amos/<subject_id>/<organ>.png`, one 3-panel
  figure (Original / Ground Truth / Prediction, with Dice in the title) per (subject, organ) —
  130 PNGs total, same style as the Text3DSAM/Medical-SAM3 figures already in this repo
- **Raw predicted masks** — `results/masks_voxtell_amos/<organ>/<subject_id>.npz` (key:
  `segs`, a boolean 3D array) — kept in case you need the full mask later, not just the Dice
- Re-running resumes automatically: completed (subject, organ) pairs already in the CSV are
  skipped, so "Save Version" mid-run never loses finished work.


In [ ]:
def run_amos22_mri():
    fig_root = RESULTS_ROOT / 'results_voxtell_amos'
    fig_root.mkdir(parents=True, exist_ok=True)
    masks_root = RESULTS_ROOT / 'masks_voxtell_amos'
    masks_root.mkdir(parents=True, exist_ok=True)
    csv_path = RESULTS_ROOT / 'predictions_voxtell_amos22_mri.csv'

    fieldnames = ['subject_id', 'split', 'organ', 'label_id', 'prompt_text',
                  'volume_shape', 'n_pred_voxels', 'dice', 'inference_seconds',
                  'pred_centroid_vox_0', 'pred_centroid_vox_1', 'pred_centroid_vox_2',
                  'pred_bbox_vox_0min', 'pred_bbox_vox_1min', 'pred_bbox_vox_2min',
                  'pred_bbox_vox_0max', 'pred_bbox_vox_1max', 'pred_bbox_vox_2max',
                  'pred_side_axis0', 'pred_side_axis1', 'pred_side_axis2',
                  'gt_centroid_vox_0', 'gt_centroid_vox_1', 'gt_centroid_vox_2',
                  'gt_side_axis0', 'gt_side_axis1', 'gt_side_axis2']

    already_done = set()
    write_header = not csv_path.exists()
    if csv_path.exists():
        with open(csv_path, newline='') as f:
            for row in csv.DictReader(f):
                already_done.add((row['subject_id'], row['organ']))
        print(f'Resuming: found {len(already_done)} already completed (subject, organ) cases in {csv_path.name}')

    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
            f.flush()

        for si, (subject_id, rows) in enumerate(all_subjects.items(), 1):
            organs = [r['organ'] for r in rows]
            pending = [r for r in rows if (subject_id, r['organ']) not in already_done]
            if not pending:
                print(f'[{si}/{len(all_subjects)}] {subject_id}: already completed, skipping')
                continue

            image_path = AMOS_DIR / rows[0]['image_path']
            label_path = AMOS_DIR / rows[0]['label_path']

            print(f'[{si}/{len(all_subjects)}] {subject_id}: predicting {len(organs)} organs in one call...')
            t0 = time.time()
            img_arr, props = load_volume_ras(image_path)  # raw (1, X, Y, Z), what the model expects
            img_spatial = spatial_view(img_arr)  # (X, Y, Z) -- for slicing/plotting only, never fed to the model
            try:
                pred_all = predictor.predict_single_image(img_arr, organs)  # -> (n_organs, X, Y, Z), already binary 0/1
            except Exception as e:
                print(f'  !! prediction failed for {subject_id}: {e}')
                continue
            elapsed_total = time.time() - t0
            print(f'  done in {elapsed_total:.1f}s ({elapsed_total / len(organs):.2f}s/organ average)')

            label_vol, _ = load_seg_ras(label_path)

            for oi, row0 in enumerate(rows):
                organ = row0['organ']
                if (subject_id, organ) in already_done:
                    continue
                label_id = int(row0['label_id'])

                pred_mask = pred_all[oi].astype(bool)  # already binary 0/1 from predict_single_image, no thresholding needed
                gt_mask = label_vol == label_id
                dice = dice_score(pred_mask, gt_mask)
                n_voxels = int(pred_mask.sum())
                gt = mask_ground_truth_3d(pred_mask) if n_voxels > 0 else None

                out_row = {
                    'subject_id': subject_id, 'split': row0['split'], 'organ': organ,
                    'label_id': label_id, 'prompt_text': row0['prompt_text'],
                    'volume_shape': row0['volume_shape'], 'n_pred_voxels': n_voxels,
                    'dice': round(dice, 4), 'inference_seconds': round(elapsed_total / len(organs), 2),
                    'pred_centroid_vox_0': None, 'pred_centroid_vox_1': None, 'pred_centroid_vox_2': None,
                    'pred_bbox_vox_0min': None, 'pred_bbox_vox_1min': None, 'pred_bbox_vox_2min': None,
                    'pred_bbox_vox_0max': None, 'pred_bbox_vox_1max': None, 'pred_bbox_vox_2max': None,
                    'pred_side_axis0': None, 'pred_side_axis1': None, 'pred_side_axis2': None,
                    'gt_centroid_vox_0': row0['gt_centroid_vox_0'], 'gt_centroid_vox_1': row0['gt_centroid_vox_1'],
                    'gt_centroid_vox_2': row0['gt_centroid_vox_2'], 'gt_side_axis0': row0['gt_side_axis0'],
                    'gt_side_axis1': row0['gt_side_axis1'], 'gt_side_axis2': row0['gt_side_axis2'],
                }
                if gt is not None:
                    (cx, cy, cz), bbox, (s0, s1, s2) = gt
                    out_row.update({
                        'pred_centroid_vox_0': cx, 'pred_centroid_vox_1': cy, 'pred_centroid_vox_2': cz,
                        'pred_bbox_vox_0min': bbox[0], 'pred_bbox_vox_1min': bbox[1], 'pred_bbox_vox_2min': bbox[2],
                        'pred_bbox_vox_0max': bbox[3], 'pred_bbox_vox_1max': bbox[4], 'pred_bbox_vox_2max': bbox[5],
                        'pred_side_axis0': s0, 'pred_side_axis1': s1, 'pred_side_axis2': s2,
                    })
                writer.writerow(out_row)
                f.flush()
                already_done.add((subject_id, organ))
                print(f'    {organ}: dice={dice:.3f}, pred_voxels={n_voxels}')

                organ_mask_dir = masks_root / struct_slug(organ)
                organ_mask_dir.mkdir(parents=True, exist_ok=True)
                np.savez_compressed(organ_mask_dir / f'{subject_id}.npz', segs=pred_mask)

                # NOTE: slice_axis/take_slice use img_spatial (3D), NOT the raw 4D img_arr fed to
                # the model above -- np.argmin(img_arr.shape) would wrongly pick the channel axis (size 1).
                slice_axis = int(np.argmin(img_spatial.shape))
                gt_coords = np.nonzero(gt_mask)
                slice_idx = int(round(gt_coords[slice_axis].mean())) if len(gt_coords[0]) else img_spatial.shape[slice_axis] // 2
                def take_slice(vol, ax=slice_axis, idx=slice_idx):
                    sl = [slice(None)] * 3; sl[ax] = idx
                    return vol[tuple(sl)].T
                subj_fig_dir = fig_root / subject_id
                subj_fig_dir.mkdir(parents=True, exist_ok=True)
                save_panel_figure(
                    subj_fig_dir / f'{struct_slug(organ)}.png',
                    row0['prompt_text'], dice,
                    window(take_slice(img_spatial)), take_slice(gt_mask), take_slice(pred_mask),
                )

    print(f'all done -> {csv_path}, {fig_root}')

run_amos22_mri()


## 11. Organ-wise visualization (grid across all subjects, one figure per organ)

The per-(subject, organ) panels from the full run above are grouped **by subject**
(`results_voxtell_amos/<subject_id>/<organ>.png`) — the same layout the Medical-SAM3
notebook uses. This section adds the other grouping: **one figure per organ**, showing every
subject side by side, so you can compare how a single organ performs across your whole
10-subject sample at a glance (matches `scripts/visualize_text3dsam_predictions.py`'s style,
generalized from one subject to one organ). Also adds a mean-Dice-per-organ bar chart.

Reuses the raw `.npz` masks already saved under `masks_voxtell_amos/` — no re-running
VoxTell needed.

In [ ]:
import math

organwise_root = RESULTS_ROOT / 'results_voxtell_amos_organwise'
organwise_root.mkdir(parents=True, exist_ok=True)

pred_df = pd.read_csv(RESULTS_ROOT / 'predictions_voxtell_amos22_mri.csv')
masks_root = RESULTS_ROOT / 'masks_voxtell_amos'

def render_organ_grid(organ):
    organ_rows = pred_df[pred_df['organ'] == organ].sort_values('subject_id')
    n = len(organ_rows)
    ncols = 5
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for i, (_, row) in enumerate(organ_rows.iterrows()):
        subject_id = row['subject_id']
        subj_manifest_rows = all_subjects[subject_id]
        row0 = next(r for r in subj_manifest_rows if r['organ'] == organ)
        image_path = AMOS_DIR / row0['image_path']
        label_path = AMOS_DIR / row0['label_path']

        img_arr = spatial_view(load_volume_ras(image_path)[0])  # this cell only plots, never feeds the model
        label_vol, _ = load_seg_ras(label_path)
        gt_mask = label_vol == int(row0['label_id'])

        mask_npz = masks_root / struct_slug(organ) / f'{subject_id}.npz'
        pred_mask = np.load(mask_npz)['segs'].astype(bool) if mask_npz.exists() else np.zeros_like(gt_mask)

        slice_axis = int(np.argmin(img_arr.shape))
        gt_coords = np.nonzero(gt_mask)
        slice_idx = int(round(gt_coords[slice_axis].mean())) if len(gt_coords[0]) else img_arr.shape[slice_axis] // 2

        def take_slice(vol, ax=slice_axis, idx=slice_idx):
            sl = [slice(None)] * 3; sl[ax] = idx
            return vol[tuple(sl)]

        ax = axes[i]
        ax.imshow(window(take_slice(img_arr)).T, cmap='gray', origin='lower')
        gt_slice = take_slice(gt_mask)
        pred_slice = take_slice(pred_mask)
        if gt_slice.any():
            ax.contour(gt_slice.T, levels=[0.5], colors='lime', linewidths=1.5)
        if pred_slice.any():
            ax.contour(pred_slice.T, levels=[0.5], colors='red', linewidths=1.5)
        ax.set_title(f"{subject_id}\ndice={row['dice']:.2f}", fontsize=10)
        ax.axis('off')

    for j in range(n, len(axes)):
        axes[j].axis('off')

    fig.suptitle(f'VoxTell -- "{organ}" across all subjects  (green=ground truth, red=prediction)', fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    out_path = organwise_root / f'{struct_slug(organ)}.png'
    fig.savefig(out_path, dpi=130)
    plt.close(fig)
    return out_path

for organ in pred_df['organ'].unique():
    p = render_organ_grid(organ)
    print('wrote', p)

# Mean Dice per organ bar chart, same style as Text3DSAM's scripts/visualize_text3dsam_predictions.py
summary = pred_df.groupby('organ')['dice'].mean().sort_values(ascending=False)
fig2, ax2 = plt.subplots(figsize=(9, 5))
bars = ax2.bar(summary.index, summary.values, color='#4C72B0')
ax2.set_ylabel(f'Mean Dice ({pred_df["subject_id"].nunique()} subjects)')
ax2.set_title('VoxTell mean Dice per organ -- AMOS22 MRI')
ax2.set_ylim(0, max(0.05, summary.values.max() * 1.15))
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')
for b, v in zip(bars, summary.values):
    ax2.text(b.get_x() + b.get_width() / 2, v + 0.005, f'{v:.2f}', ha='center', va='bottom', fontsize=8)
fig2.tight_layout()
bar_path = organwise_root / 'voxtell_dice_per_organ.png'
fig2.savefig(bar_path, dpi=130)
plt.close(fig2)
print('wrote', bar_path)


## 12. Summary & Output Archiving

In [ ]:
import shutil
import pandas as pd

csv_path = RESULTS_ROOT / 'predictions_voxtell_amos22_mri.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print('=== AMOS22 MRI (VoxTell) Summary ===')
    print(f'Total evaluated cases: {len(df)}')
    print(f"Mean Dice: {df['dice'].mean():.4f}")
    print()
    print('Mean Dice by organ:')
    print(df.groupby('organ')['dice'].mean().sort_values(ascending=False).round(4))

zip_path = WORKING_DIR / 'voxtell_amos22_mri_results'
shutil.make_archive(str(zip_path), 'zip', str(RESULTS_ROOT))
print(f"\nArchive created at {zip_path}.zip ({os.path.getsize(str(zip_path) + '.zip') / 1e6:.1f} MB)")
print("This zip file will appear directly in Kaggle's Output panel.")
